In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import HillClimbSearch, BIC, BayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

os.makedirs("bayesian_net", exist_ok=True)

In [2]:
CLASSIFIED_DIR = "clean_crop_contribution_data/aggregated_data/kmeans_classes"
FEATURES_FILE  = "engineered_climate_features_slight_correlation.csv"
MIN_ROWS       = 200
N_SPLITS       = 5
N_BINS         = 4   # discretization bins for continuous climate features

In [3]:
features_df = pd.read_csv(FEATURES_FILE)
csv_files   = glob.glob(os.path.join(CLASSIFIED_DIR, "*_classified.csv"))
print(f"Found {len(csv_files)} classified files")

Found 54 classified files


In [4]:
all_crop_summary = []

for csv_path in csv_files:
    crop_name = os.path.basename(csv_path).replace("_classified.csv", "")
    print(f"\n{'='*60}\nCROP: {crop_name}\n{'='*60}")

    crop_dir = os.path.join("bayesian_net", f"{crop_name}_bn")
    os.makedirs(crop_dir, exist_ok=True)

    class_df = pd.read_csv(csv_path)
    if len(class_df) <= MIN_ROWS:
        print(f"  Skipping - only {len(class_df)} rows")
        continue

    class_df["location"] = class_df["State"] + "_" + class_df["District"]
    merged_df = class_df.merge(features_df, on="location", how="inner")
    print(f"  Merged shape: {merged_df.shape}")

    # Drop rare classes
    class_counts = merged_df["Yield_Class"].value_counts()
    valid_classes = class_counts[class_counts >= N_SPLITS].index.tolist()
    merged_df = merged_df[merged_df["Yield_Class"].isin(valid_classes)].copy()

    if len(merged_df) <= MIN_ROWS or len(valid_classes) < 2:
        print(f"  Skipping - insufficient data after class filter")
        continue

    print(f"  Class distribution:\n{merged_df['Yield_Class'].value_counts().to_string()}")

    # ── Discretize continuous features ──────────────────────────────────────
    drop_cols = ["State", "District", "location", "Median", "Max", "Yield_Class"]
    feature_cols = [c for c in merged_df.columns if c not in drop_cols]

    disc = KBinsDiscretizer(n_bins=N_BINS, encode="ordinal", strategy="quantile")
    X_disc = disc.fit_transform(merged_df[feature_cols])
    X_disc = pd.DataFrame(X_disc, columns=feature_cols,
                          index=merged_df.index).astype(int).astype(str)
    X_disc["Yield_Class"] = merged_df["Yield_Class"].values

    # ── Cross-validation ─────────────────────────────────────────────────────
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    y  = merged_df["Yield_Class"].values
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_disc, y), 1):
        train_df = X_disc.iloc[train_idx].reset_index(drop=True)
        test_df  = X_disc.iloc[test_idx].reset_index(drop=True)

        # Structure learning on training fold
        hc    = HillClimbSearch(train_df)
        best_model = hc.estimate(
            scoring_method=BIC(train_df),
            max_indegree=3,        # keep it tractable
            max_iter=int(1e4)
        )

        model = DiscreteBayesianNetwork(best_model.edges())
        # Ensure Yield_Class is in the network even if isolated
        if "Yield_Class" not in model.nodes():
            model.add_node("Yield_Class")

        model.fit(train_df, estimator=BayesianEstimator,
                  prior_type="BDeu", equivalent_sample_size=10)

        # ── Inference ────────────────────────────────────────────────────────
        infer   = VariableElimination(model)
        y_pred  = []
        y_true  = test_df["Yield_Class"].tolist()
        evidence_cols = [c for c in feature_cols if c in model.nodes()]

        for _, row in test_df.iterrows():
            evidence = {col: row[col] for col in evidence_cols}
            try:
                q = infer.map_query(variables=["Yield_Class"], evidence=evidence,
                                    show_progress=False)
                y_pred.append(q["Yield_Class"])
            except Exception:
                # Fall back to majority class if inference fails
                y_pred.append(train_df["Yield_Class"].mode()[0])

        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
        fold_metrics.append({"acc": acc, "f1": f1})
        print(f"  Fold {fold}: acc={acc:.4f}  f1={f1:.4f}")

    acc_scores = [m["acc"] for m in fold_metrics]
    f1_scores  = [m["f1"]  for m in fold_metrics]

    # ── Refit on full data & save structure plot ─────────────────────────────
    hc_full    = HillClimbSearch(X_disc)
    best_full  = hc_full.estimate(scoring_method=BIC(X_disc),
                               max_indegree=3, max_iter=int(1e4))
    model_full = DiscreteBayesianNetwork(best_full.edges())
    model_full.fit(X_disc, estimator=BayesianEstimator,
                   prior_type="BDeu", equivalent_sample_size=10)

    # Plot network
    G   = nx.DiGraph(model_full.edges())
    pos = nx.spring_layout(G, seed=42)
    node_colors = ["#e74c3c" if n == "Yield_Class" else "#3498db" for n in G.nodes()]
    fig, ax = plt.subplots(figsize=(10, 7))
    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors,
                     node_size=1200, font_size=7, arrows=True,
                     edge_color="gray", arrowsize=15)
    ax.set_title(f"{crop_name} — Bayesian network structure")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(crop_dir, "bn_structure.png"), dpi=120, bbox_inches="tight")
    plt.close()

    # Which features are direct parents of Yield_Class?
    parents_of_yield = list(model_full.get_parents("Yield_Class")) \
                       if "Yield_Class" in model_full.nodes() else []
    print(f"  Direct parents of Yield_Class: {parents_of_yield}")

    summary = {
        "crop":         crop_name,
        "n_rows":       len(merged_df),
        "n_classes":    len(valid_classes),
        "acc_mean":     np.mean(acc_scores),
        "acc_std":      np.std(acc_scores),
        "f1_mean":      np.mean(f1_scores),
        "f1_std":       np.std(f1_scores),
        "yield_parents": ", ".join(parents_of_yield),
        "n_edges":      len(model_full.edges()),
    }
    all_crop_summary.append(summary)
    print(f"  acc={summary['acc_mean']:.4f}±{summary['acc_std']:.4f}  "
          f"f1={summary['f1_mean']:.4f}±{summary['f1_std']:.4f}")

c:\Users\Atharva Jagtap\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 23 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Atharva Jagtap\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 24 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Atharva Jagtap\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 27 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Atharva Jagtap\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e


CROP: arecanut
  Skipping - only 152 rows

CROP: arhar_tur
  Merged shape: (665, 42)
  Class distribution:
Yield_Class
Medium    293
High      275
Low        97


  0%|          | 0/10000 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'tp_kharif_mean': 'C', 'tp_kharif_std': 'C', 'e_kharif_mean': 'C', 'e_kharif_std': 'C', 'swvl1_kharif_mean': 'C', 'soil_air_diff_kharif_mean': 'C', 'soil_air_diff_kharif_std': 'C', 'runoff_ratio_kharif_mean': 'C', 'e_rabi_mean': 'C', 'e_rabi_std': 'C', 'swvl1_rabi_mean': 'C', 'swvl1_rabi_std': 'C', 'soil_air_diff_rabi_mean': 'C', 'soil_air_diff_rabi_std': 'C', 'runoff_ratio_rabi_mean': 'C', 'tp_zaid_mean': 'C', 'e_zaid_mean': 'C', 'e_zaid_std': 'C', 'swvl1_zaid_std': 'C', 'soil_air_diff_zaid_mean': 'C', 'runoff_ratio_zaid_mean': 'C', 'tp_kharif_fraction': 'C', 't2m_amplitude': 'C', 't2m_peak_sin': 'C', 't2m_peak_cos': 'C', 'GDD_kharif': 'C', 'GDD_rabi': 'C', 'heat_months_30C': 'C', 'heat_months_35C': 'C', 'humidity_variability': 'C', 'swvl1_amplitude': 'C', 'swvl1_stability': 'C', 'soil_moisture_seasonality': 'C', 'cvh': 'C', 'water_retention_index': 'C', 'humidity_stress_index': '

  Fold 1: acc=0.5038  f1=0.3473


...

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'tp_kharif_mean': 'C', 'tp_kharif_std': 'C', 'e_kharif_mean': 'C', 'e_kharif_std': 'C', 'swvl1_kharif_mean': 'C', 'soil_air_diff_kharif_mean': 'C', 'soil_air_diff_kharif_std': 'C', 'runoff_ratio_kharif_mean': 'C', 'e_rabi_mean': 'C', 'e_rabi_std': 'C', 'swvl1_rabi_mean': 'C', 'swvl1_rabi_std': 'C', 'soil_air_diff_rabi_mean': 'C', 'soil_air_diff_rabi_std': 'C', 'runoff_ratio_rabi_mean': 'C', 'tp_zaid_mean': 'C', 'e_zaid_mean': 'C', 'e_zaid_std': 'C', 'swvl1_zaid_std': 'C', 'soil_air_diff_zaid_mean': 'C', 'runoff_ratio_zaid_mean': 'C', 'tp_kharif_fraction': 'C', 't2m_amplitude': 'C', 't2m_peak_sin': 'C', 't2m_peak_cos': 'C', 'GDD_kharif': 'C', 'GDD_rabi': 'C', 'heat_months_30C': 'C', 'heat_months_35C': 'C', 'humidity_variability': 'C', 'swvl1_amplitude': 'C', 'swvl1_stability': 'C', 'soil_moisture_seasonality': 'C', 'cvh': 'C', 'water_retention_index': 'C', 'humidity_stress_index': '

  Fold 5: acc=0.6032  f1=0.5967


  0%|          | 0/10000 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'tp_kharif_mean': 'C', 'tp_kharif_std': 'C', 'e_kharif_mean': 'C', 'e_kharif_std': 'C', 'swvl1_kharif_mean': 'C', 'soil_air_diff_kharif_mean': 'C', 'soil_air_diff_kharif_std': 'C', 'runoff_ratio_kharif_mean': 'C', 'e_rabi_mean': 'C', 'e_rabi_std': 'C', 'swvl1_rabi_mean': 'C', 'swvl1_rabi_std': 'C', 'soil_air_diff_rabi_mean': 'C', 'soil_air_diff_rabi_std': 'C', 'runoff_ratio_rabi_mean': 'C', 'tp_zaid_mean': 'C', 'e_zaid_mean': 'C', 'e_zaid_std': 'C', 'swvl1_zaid_std': 'C', 'soil_air_diff_zaid_mean': 'C', 'runoff_ratio_zaid_mean': 'C', 'tp_kharif_fraction': 'C', 't2m_amplitude': 'C', 't2m_peak_sin': 'C', 't2m_peak_cos': 'C', 'GDD_kharif': 'C', 'GDD_rabi': 'C', 'heat_months_30C': 'C', 'heat_months_35C': 'C', 'humidity_variability': 'C', 'swvl1_amplitude': 'C', 'swvl1_stability': 'C', 'soil_moisture_seasonality': 'C', 'cvh': 'C', 'water_retention_index': 'C', 'humidity_stress_index': '

  Direct parents of Yield_Class: ['swvl1_rabi_std', 'GDD_kharif']
  acc=0.6063±0.0597  f1=0.6074±0.0562


In [5]:
summary_df = pd.DataFrame(all_crop_summary).sort_values("f1_mean", ascending=False)

print("\n" + "="*60)
print("CROSS-CROP SUMMARY — BAYESIAN NETWORK")
print("="*60)
cols = ["crop", "n_rows", "n_classes", "acc_mean", "acc_std",
        "f1_mean", "f1_std", "yield_parents", "n_edges"]
print(summary_df[cols].to_string(index=False))

print("\nOverall average:")
print(f"  Accuracy : {summary_df['acc_mean'].mean():.4f}, SD: {summary_df['acc_std'].mean():.4f}")
print(f"  F1       : {summary_df['f1_mean'].mean():.4f}, SD: {summary_df['f1_std'].mean():.4f}")

summary_df.to_csv("bayesian_net/all_crops_summary_bn.csv", index=False)
print("\nSaved -> bayesian_net/all_crops_summary_bn.csv")


CROSS-CROP SUMMARY — BAYESIAN NETWORK
                crop  n_rows  n_classes  acc_mean  acc_std  f1_mean   f1_std                                  yield_parents  n_edges
           groundnut     583          2  0.744371 0.035685 0.705664 0.048419                                       GDD_rabi       53
              garlic     439          3  0.644645 0.032445 0.638476 0.030327            t2m_peak_sin, water_retention_index       51
               onion     573          3  0.687582 0.025172 0.631273 0.055358            tp_zaid_mean, water_retention_index       56
                ragi     379          3  0.619965 0.054809 0.616596 0.046369        t2m_peak_sin, soil_moisture_seasonality       47
               wheat     630          3  0.606349 0.059730 0.607406 0.056232                     swvl1_rabi_std, GDD_kharif       57
        sweet_potato     457          3  0.693502 0.037736 0.567075 0.060334   runoff_ratio_rabi_mean, humidity_variability       52
rapeseed_and_mustard     656  